# RAG Pipeline — Policy & Historical Support Knowledge

This notebook builds and validates the two FAISS knowledge stores used by the application.

**Authoritative source**
- Banking policy documents in `data/policies`

**Secondary source**
- Historical support-ticket cases from `data/support_tickets.csv`

The QA pairs are used separately for evaluation and are not added to the primary retrieval corpus.


## 1. Setup


In [1]:
from pathlib import Path
import pickle
import re

import faiss
import numpy as np
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd().resolve().parent

POLICY_DIR = PROJECT_ROOT / "data" / "policies"
TICKETS_PATH = PROJECT_ROOT / "data" / "support_tickets.csv"
VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store"

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Policy directory:", POLICY_DIR)
print("Policy files:", list(POLICY_DIR.glob("*.txt")))


e:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: E:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System
Policy directory: E:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System\data\policies
Policy files: [WindowsPath('E:/HCL Guvi/Banking Support & Fraud/Banking Support & Fraud Intelligence System/data/policies/account_access_policy.txt'), WindowsPath('E:/HCL Guvi/Banking Support & Fraud/Banking Support & Fraud Intelligence System/data/policies/fraud_handling_policy.txt'), WindowsPath('E:/HCL Guvi/Banking Support & Fraud/Banking Support & Fraud Intelligence System/data/policies/kyc_policy.txt'), WindowsPath('E:/HCL Guvi/Banking Support & Fraud/Banking Support & Fraud Intelligence System/data/policies/loan_processing_policy.txt'), WindowsPath('E:/HCL Guvi/Banking Support & Fraud/Banking Support & Fraud Intelligence System/data/policies/refund_dispute_policy.txt')]


## 2. Load and Inspect Policy Documents


In [2]:
policy_documents = []

for policy_path in sorted(POLICY_DIR.glob("*.txt")):
    text = policy_path.read_text(encoding="utf-8")
    policy_documents.append(
        {
            "source": policy_path.name,
            "text": text.strip(),
        }
    )

print("Policy documents:", len(policy_documents))

for document in policy_documents:
    print(
        f"{document['source']}: "
        f"{len(document['text'])} characters"
    )


Policy documents: 5
account_access_policy.txt: 1262 characters
fraud_handling_policy.txt: 3235 characters
kyc_policy.txt: 2515 characters
loan_processing_policy.txt: 2958 characters
refund_dispute_policy.txt: 1993 characters


## 3. Chunk Policy Documents


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

policy_chunks = []

for document in policy_documents:
    chunks = text_splitter.split_text(document["text"])

    for index, text in enumerate(chunks, start=1):
        policy_chunks.append(
            {
                "chunk_id": f"{Path(document['source']).stem}_{index}",
                "source": document["source"],
                "text": text,
            }
        )

policy_chunk_df = pd.DataFrame(policy_chunks)

print("Total policy chunks:", len(policy_chunks))
display(
    policy_chunk_df.groupby("source")
    .size()
    .rename("chunks")
    .to_frame()
)

display(policy_chunk_df.head())


Total policy chunks: 28


,chunks
source,
account_access_policy.txt,3
fraud_handling_policy.txt,8
kyc_policy.txt,6
loan_processing_policy.txt,6
refund_dispute_policy.txt,5


,chunk_id,source,text
0,account_access_policy_1,account_access_policy.txt,ACCOUNT ACCESS AND GENERAL ACCOUNT OPERATIONS ...
1,account_access_policy_2,account_access_policy.txt,2.2 Customers should first check with their HR...
2,account_access_policy_3,account_access_policy.txt,3.2 Required documents include:\n - PAN\n ...
3,fraud_handling_policy_1,fraud_handling_policy.txt,FRAUD HANDLING POLICY — RETAIL BANKING\nVersio...
4,fraud_handling_policy_2,fraud_handling_policy.txt,2. DEFINITIONS\n2.1 Unauthorized Transaction: ...


## 4. Generate Policy Embeddings


In [4]:
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

policy_texts = [
    chunk["text"]
    for chunk in policy_chunks
]

policy_embeddings = embedding_model.encode(
    policy_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

policy_embeddings = np.asarray(
    policy_embeddings,
    dtype="float32",
)

print("Embedding shape:", policy_embeddings.shape)
print("Embedding dimension:", policy_embeddings.shape[1])


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.25it/s]

Embedding shape: (28, 384)
Embedding dimension: 384


## 5. Build and Save Policy FAISS Index


In [5]:
policy_index = faiss.IndexFlatIP(
    policy_embeddings.shape[1]
)

policy_index.add(policy_embeddings)

POLICY_INDEX_PATH = VECTOR_STORE_DIR / "policy.index"
POLICY_METADATA_PATH = VECTOR_STORE_DIR / "policy_metadata.pkl"

faiss.write_index(
    policy_index,
    str(POLICY_INDEX_PATH),
)

with open(POLICY_METADATA_PATH, "wb") as file:
    pickle.dump(policy_chunks, file)

print("Policy vectors:", policy_index.ntotal)
print("Policy index:", POLICY_INDEX_PATH)
print("Policy metadata:", POLICY_METADATA_PATH)


Policy vectors: 28
Policy index: E:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System\vector_store\policy.index
Policy metadata: E:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System\vector_store\policy_metadata.pkl


## 6. Build Historical Support-Ticket Corpus


In [6]:
tickets = pd.read_csv(TICKETS_PATH)

ticket_rag = (
    tickets[
        [
            "category",
            "sub_category",
            "query_text",
            "resolution_text",
        ]
    ]
    .drop_duplicates(
        subset=["query_text", "resolution_text"]
    )
    .reset_index(drop=True)
)

print("Original tickets:", len(tickets))
print("Unique historical cases:", len(ticket_rag))

display(
    ticket_rag["category"]
    .value_counts()
    .rename("cases")
    .to_frame()
)


Original tickets: 200
Unique historical cases: 81


,cases
category,
Fraud/Unauthorized,32
Loan,24
Account Access,15
KYC,10


In [7]:
historical_documents = []

for index, row in ticket_rag.iterrows():

    resolution = re.sub(
        r"\s*\(Ref\s*#?[A-Za-z0-9-]+\)",
        "",
        str(row["resolution_text"]),
        flags=re.IGNORECASE,
    )

    text = (
        f"Category: {row['category']}\n\n"
        f"Customer Query:\n{row['query_text']}\n\n"
        f"Historical Resolution:\n{resolution}"
    )

    historical_documents.append(
        {
            "chunk_id": f"ticket_case_{index + 1}",
            "source": "support_tickets",
            "category": row["category"],
            "text": text.strip(),
        }
    )

print("Historical documents:", len(historical_documents))
display(pd.DataFrame(historical_documents).head())


Historical documents: 81


,chunk_id,source,category,text
0,ticket_case_1,support_tickets,Fraud/Unauthorized,Category: Fraud/Unauthorized\n\nCustomer Query...
1,ticket_case_2,support_tickets,Fraud/Unauthorized,Category: Fraud/Unauthorized\n\nCustomer Query...
2,ticket_case_3,support_tickets,Fraud/Unauthorized,Category: Fraud/Unauthorized\n\nCustomer Query...
3,ticket_case_4,support_tickets,Fraud/Unauthorized,Category: Fraud/Unauthorized\n\nCustomer Query...
4,ticket_case_5,support_tickets,Fraud/Unauthorized,Category: Fraud/Unauthorized\n\nCustomer Query...


## 7. Generate Historical Embeddings and Build FAISS Index


In [8]:
historical_texts = [
    document["text"]
    for document in historical_documents
]

historical_embeddings = embedding_model.encode(
    historical_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

historical_embeddings = np.asarray(
    historical_embeddings,
    dtype="float32",
)

historical_index = faiss.IndexFlatIP(
    historical_embeddings.shape[1]
)

historical_index.add(
    historical_embeddings
)

TICKET_INDEX_PATH = VECTOR_STORE_DIR / "support_tickets.index"
TICKET_METADATA_PATH = VECTOR_STORE_DIR / "support_ticket_metadata.pkl"

faiss.write_index(
    historical_index,
    str(TICKET_INDEX_PATH),
)

with open(TICKET_METADATA_PATH, "wb") as file:
    pickle.dump(
        historical_documents,
        file,
    )

print("Historical vectors:", historical_index.ntotal)
print("Historical index:", TICKET_INDEX_PATH)
print("Historical metadata:", TICKET_METADATA_PATH)


Batches: 100%|██████████| 3/3 [00:00<00:00,  7.62it/s]

Historical vectors: 81
Historical index: E:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System\vector_store\support_tickets.index
Historical metadata: E:\HCL Guvi\Banking Support & Fraud\Banking Support & Fraud Intelligence System\vector_store\support_ticket_metadata.pkl


## 8. Retrieval Smoke Tests


In [9]:
def retrieve_from_index(
    query,
    index,
    metadata,
    top_k=3,
    category=None,
    allowed_sources=None,
):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        len(metadata),
    )

    results = []

    for score, index_id in zip(
        scores[0],
        indices[0],
    ):
        if index_id == -1:
            continue

        item = metadata[index_id]

        if (
            allowed_sources is not None
            and item["source"] not in allowed_sources
        ):
            continue

        if (
            category is not None
            and item.get("category") != category
        ):
            continue

        results.append(
            {
                "source": item["source"],
                "chunk_id": item["chunk_id"],
                "category": item.get("category"),
                "similarity_score": round(
                    float(score), 4
                ),
                "text": item["text"],
            }
        )

        if len(results) >= top_k:
            break

    return results


In [10]:
# Policy retrieval tests
policy_index_loaded = faiss.read_index(
    str(POLICY_INDEX_PATH)
)

with open(POLICY_METADATA_PATH, "rb") as file:
    policy_metadata_loaded = pickle.load(file)

test_queries = [
    (
        "Fraud/Unauthorized",
        "I see an unauthorized transaction on my account.",
        [
            "fraud_handling_policy.txt",
            "refund_dispute_policy.txt",
        ],
    ),
    (
        "KYC",
        "What documents do I need for KYC?",
        ["kyc_policy.txt"],
    ),
    (
        "Loan",
        "What documents do I need for a home loan?",
        ["loan_processing_policy.txt"],
    ),
    (
        "Account Access",
        "I'm locked out of my net banking.",
        ["account_access_policy.txt"],
    ),
]

for intent, query, sources in test_queries:

    print("=" * 70)
    print("Intent:", intent)
    print("Query:", query)

    results = retrieve_from_index(
        query=query,
        index=policy_index_loaded,
        metadata=policy_metadata_loaded,
        top_k=3,
        allowed_sources=sources,
    )

    for result in results:
        print(
            f"{result['source']} | "
            f"{result['chunk_id']} | "
            f"{result['similarity_score']}"
        )


Intent: Fraud/Unauthorized
Query: I see an unauthorized transaction on my account.
fraud_handling_policy.txt | fraud_handling_policy_2 | 0.7455
fraud_handling_policy.txt | fraud_handling_policy_3 | 0.6824
fraud_handling_policy.txt | fraud_handling_policy_5 | 0.6504
Intent: KYC
Query: What documents do I need for KYC?
kyc_policy.txt | kyc_policy_2 | 0.7176
kyc_policy.txt | kyc_policy_3 | 0.7042
kyc_policy.txt | kyc_policy_5 | 0.654
Intent: Loan
Query: What documents do I need for a home loan?
loan_processing_policy.txt | loan_processing_policy_3 | 0.7599
loan_processing_policy.txt | loan_processing_policy_2 | 0.6584
loan_processing_policy.txt | loan_processing_policy_4 | 0.6347
Intent: Account Access
Query: I'm locked out of my net banking.
account_access_policy.txt | account_access_policy_1 | 0.7278
account_access_policy.txt | account_access_policy_2 | 0.5625
account_access_policy.txt | account_access_policy_3 | 0.5365


In [11]:
# Historical retrieval test
ticket_index_loaded = faiss.read_index(
    str(TICKET_INDEX_PATH)
)

with open(TICKET_METADATA_PATH, "rb") as file:
    ticket_metadata_loaded = pickle.load(file)

results = retrieve_from_index(
    query="I forgot my ATM PIN and cannot reset it online.",
    index=ticket_index_loaded,
    metadata=ticket_metadata_loaded,
    top_k=3,
    category="Account Access",
)

for result in results:
    print("=" * 70)
    print(result["category"])
    print(result["similarity_score"])
    print(result["text"])


Account Access
0.8586
Category: Account Access

Customer Query:
I forgot my ATM PIN and can't reset it online

Historical Resolution:
ATM PIN reset link sent to registered email. Alternatively, regenerate PIN via IVR by pressing *2 after balance enquiry.
Account Access
0.6328
Category: Account Access

Customer Query:
I'm unable to login to my net banking account

Historical Resolution:
Net banking access suspended after 3 failed login attempts. Account unlocked. New IPIN to be generated via ATM or branch.
Account Access
0.5889
Category: Account Access

Customer Query:
My fixed deposit matured but amount not credited

Historical Resolution:
FD maturity amount of ₹25,000 queued for credit. Delay due to bank holiday. Amount to be credited next working day with applicable interest.


## 9. Production Retrieval Contract

The reusable production retriever in `src/rag/retriever.py` loads these saved artifacts:

- `vector_store/policy.index`
- `vector_store/policy_metadata.pkl`
- `vector_store/support_tickets.index`
- `vector_store/support_ticket_metadata.pkl`

The application keeps **authoritative policy** and **historical support cases** separate when constructing the LLM context.


## Final Result

The vector stores are ready for the production LangGraph pipeline.

No Gemini call is required in this notebook.
